<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 7 — Array and DataFrame Operations

A table is one way to represent data. An array is another.

In the last notebook, every question about Northwind could be answered in SQL **and** in pandas, with the same steps: filter, sort, join, group, window. This notebook shifts perspective. It covers operations that are **especially natural** in NumPy and pandas. SQL could express many of them too, but often only with longer, more elaborate queries.

- **NumPy** thinks in **positions and shapes**. Data sits in a grid with numbered axes, and one expression computes on thousands of values at once.
- **pandas** builds on those array ideas and adds **labels** (named rows and columns), columns of **different types**, and tools for **dates and time**.
- **SQL** relates tables through **join keys**.

Keep this sentence in mind. By the end of the notebook it should make sense:

> NumPy combines values **by position**, pandas **by label**, SQL **by join key**.

### The business in this notebook

All examples use one small, made-up coffee company: **three stores (A, B, C)** that sell **four products** (espresso, latte, muffin, bagel). The numbers are small on purpose, so you can check every result in your head.

### Four questions to ask about every operation

1. What does one input **element** or **row** represent?
2. What is the input **shape** (NumPy) or **grain** (table)?
3. What will the output shape or grain be?
4. Are values combined by **position**, by **label**, or by a **join key**?

The notebook keeps asking you to **🔮 predict** before you run a cell, usually the **shape** of the result. That habit prevents a surprising number of errors.

**How to work through it:** run the cells in order with **Shift + Enter**. Cells marked **✏️ Try it** ask you to change something. The **🧩 Exercises** are at the end of each part. Kernel: **Conda Python 3.12**.

In [ ]:
import numpy as np
import pandas as pd


def show(x):
    '''Print a value together with its shape.'''
    print(x)
    print("shape:", np.shape(x))

`show(x)` prints a value and its shape. We use it throughout, so you can compare your prediction with the real shape. `np.shape(...)` also works on plain numbers: a single number has the empty shape `()`.

---

# Part 1 — NumPy: arrays and shape

## 1.1 Shape, dimensions, and axes

Here are the unit sales of our three stores over four weeks. **One row per store, one column per week:**

|  | week 1 | week 2 | week 3 | week 4 |
|---|---:|---:|---:|---:|
| **Store A** | 10 | 12 | 15 | 11 |
| **Store B** | 8 | 9 | 14 | 13 |
| **Store C** | 20 | 18 | 17 | 22 |

As a NumPy array:

In [ ]:
sales = np.array([
    [10, 12, 15, 11],   # Store A
    [ 8,  9, 14, 13],   # Store B
    [20, 18, 17, 22],   # Store C
])

sales

The array has no idea that rows are stores and columns are weeks. **Only we know that.** NumPy only knows positions. That's why the comments in the cell matter.

Three attributes describe its structure:

In [ ]:
print("ndim: ", sales.ndim)    # number of dimensions (axes)
print("shape:", sales.shape)   # length of each axis: (rows, columns)
print("size: ", sales.size)    # total number of values

- `ndim = 2`: a **matrix**, with two axes.
- `shape = (3, 4)`: axis 0 has length 3 (stores), axis 1 has length 4 (weeks).
- `size = 12` = 3 × 4 values.

A single row of prices is a **vector**, a 1-dimensional array:

In [ ]:
prices = np.array([3.00, 4.50, 2.75, 2.25])   # espresso, latte, muffin, bagel
show(prices)

The shape of a vector is `(4,)`, a tuple with **one** number. The comma is there because Python writes a one-item tuple as `(4,)`. A vector is **not** a row and **not** a column. It has only one axis.

| Object | Example | Shape |
|---|---|---|
| single value (scalar) | `12` | `()` |
| vector | 4 prices | `(n,)` → `(4,)` |
| matrix | stores × weeks | `(rows, columns)` → `(3, 4)` |

### Axes: which direction does an operation go?

This is the most important idea in NumPy. When you add up a matrix, you must say **which axis to collapse**:

- `axis=0` works **down the rows**. It collapses the stores and leaves **one result per column** (per week).
- `axis=1` works **across the columns**. It collapses the weeks and leaves **one result per row** (per store).

```
            axis=1  ───────────►
          week1 week2 week3 week4
axis=0  A   10    12    15    11    → sum(axis=1) → 48
  │     B    8     9    14    13    →              44
  │     C   20    18    17    22    →              77
  ▼          │     │     │     │
         sum(axis=0):
            38    39    46    46
```

A trick for remembering it: **the axis you name is the one that disappears.** Shape `(3, 4)` summed over axis 0 loses the 3 and becomes `(4,)`. Summed over axis 1 it loses the 4 and becomes `(3,)`.

🔮 **Predict** the shape of each result, then run the cell.

In [ ]:
show(sales.sum())           # everything

In [ ]:
show(sales.sum(axis=0))     # collapse the stores -> one total per week

In [ ]:
show(sales.sum(axis=1))     # collapse the weeks -> one total per store

Apply the four questions to `sales.sum(axis=1)`:

1. One input element = units sold by one store in one week.
2. Input shape `(3, 4)`: stores × weeks.
3. Output shape `(3,)`: one total per store.
4. Values are combined **by position**: all numbers in the same row are added.

**✏️ Try it:** Which week sold the most in total? Use `.argmax()` (the **position** of the largest value) on the right sum. Remember that positions start at 0, so position 2 is week 3.

## 1.2 Positional indexing and slicing

NumPy picks values by **position**: `array[row, column]`, counting from 0.

### One value

In [ ]:
show(sales[0, 1])    # row 0 (Store A), column 1 (week 2)

### Slices: `start:stop:step`

A slice `start:stop` includes `start` but **excludes `stop`**, exactly like `range()` and list slicing. `:` alone means "everything along this axis".

In [ ]:
show(sales[0, 1:3])   # Store A, columns 1 and 2 (= weeks 2 and 3); column 3 is NOT included

In [ ]:
show(sales[1:4, :])   # rows 1, 2, 3 -> but there is no row 3, so we get rows 1 and 2

A slice that runs past the end is **not** an error. It stops at the last row. (A single index past the end, like `sales[3, 0]`, **is** an error.)

The third number in a slice is the **step**:

In [ ]:
show(sales[:, ::2])    # every store, every 2nd week: weeks 1 and 3
show(sales[:, -1])     # every store, the last week (negative = count from the end)

### Scalar, 1-D, or 2-D? It depends on how you index

This catches everyone at first. 🔮 **Predict** all three shapes before running the cell.

In [ ]:
show(sales[0, 0])      # a single number
print()
show(sales[:, 0])      # week 1 for every store, as an index
print()
show(sales[:, 0:1])    # week 1 for every store, as a slice

- `sales[0, 0]`: an index on **both** axes. Both axes disappear, leaving a **scalar**, shape `()`.
- `sales[:, 0]`: a single index on axis 1. That axis **disappears**, leaving a **vector**, shape `(3,)`.
- `sales[:, 0:1]`: a slice on axis 1. Slices **keep** their axis, even with length 1, leaving a **matrix** with one column, shape `(3, 1)`.

**Rule:** a single number removes an axis. A slice keeps it. The values are the same, but code that expects a column may fail with a vector, and the other way round.

**✏️ Try it:** Get Store C's sales in weeks 2 to 4 as a vector (shape `(3,)`). Then get the same numbers as a 1 × 3 matrix (shape `(1, 3)`).

## 1.3 Elementwise arithmetic

Four products, their prices, and how many of each Store A sold today:

In [ ]:
products   = np.array(["espresso", "latte", "muffin", "bagel"])
prices     = np.array([3.00, 4.50, 2.75, 2.25])
quantities = np.array([40, 25, 12, 18])

🔮 **Predict** the shape of `line_totals`:

In [ ]:
line_totals = prices * quantities
show(line_totals)

`*` multiplied **position by position**: price[0] × quantity[0], price[1] × quantity[1], and so on. One result per element, and no loop. With Python lists you would write:

In [ ]:
line_totals_loop = []
for p, q in zip(prices, quantities):
    line_totals_loop.append(p * q)
print(line_totals_loop)

Both give the same numbers. NumPy's version is shorter and, for large arrays, much faster.

All arithmetic operators work elementwise: `+ - * / ** //`, and comparisons such as `>` too:

In [ ]:
print(prices + 0.25)          # everything 25 cents more expensive
print(quantities / 2)         # half the quantities
print(line_totals > 50)       # which lines brought in more than $50?

### Elementwise `*` is not matrix multiplication `@`

`prices * quantities` returns **four** line totals. `prices @ quantities` multiplies position by position **and then adds everything up**, returning **one** number, the total revenue. That operation is called the *dot product*.

In [ ]:
show(prices * quantities)     # elementwise: 4 values
show(prices @ quantities)     # dot product: 1 value
show((prices * quantities).sum())   # the same as the line above

If you mean "one result per element", use `*`. Use `@` only when you really want the multiply-and-add of linear algebra.

## 1.4 Broadcasting

Elementwise operations need values to pair up. What if the two sides have **different shapes**?

### A scalar with an array

The simplest case: a single number combined with an array. The number is used for **every** element, as if it had been copied to the array's shape:

In [ ]:
show(sales * 1.10)    # every store, every week: 10% more

### A vector with a matrix

Now a units table of **stores × products**, and the prices of the four products:

In [ ]:
units = np.array([
    # espresso latte muffin bagel
    [40,       25,   12,    18],    # Store A
    [35,       30,   10,    20],    # Store B
    [60,       45,   25,    15],    # Store C
])
prices = np.array([3.00, 4.50, 2.75, 2.25])

print("units: ", units.shape)
print("prices:", prices.shape)

🔮 **Predict** the shape of `revenue`. Can a `(3, 4)` array be multiplied by a `(4,)` array?

In [ ]:
revenue = units * prices
show(revenue)

NumPy **stretched** the price vector down the rows. Each store's row was multiplied by the same four prices:

```
units (3, 4)        prices (4,)            revenue (3, 4)
[40 25 12 18]   *   [3.00 4.50 2.75 2.25]  =  [120.  112.5  33.   40.5 ]
[35 30 10 20]   *   [3.00 4.50 2.75 2.25]  =  [105.  135.   27.5  45.  ]
[60 45 25 15]   *   [3.00 4.50 2.75 2.25]  =  [180.  202.5  68.75 33.75]
```

No copy of `prices` is actually made. NumPy just behaves as if it were. This is called **broadcasting**.

The same works for costs. Each product costs a fixed amount to make:

In [ ]:
cost_per_product = np.array([0.80, 1.20, 1.10, 0.90])   # espresso, latte, muffin, bagel

net = revenue - units * cost_per_product
show(net)

### The broadcasting rule

NumPy compares the two shapes **from the right** (the last axis first). Each pair of lengths must either be **equal**, or one of them must be **1**. A missing axis counts as 1.

| left shape | right shape | compare from the right | works? | result |
|---|---|---|---|---|
| `(3, 4)` | `()` scalar | nothing to compare | ✅ | `(3, 4)` |
| `(3, 4)` | `(4,)` | 4 vs 4 ✅, 3 vs (missing) ✅ | ✅ | `(3, 4)` |
| `(3, 4)` | `(3,)` | 4 vs **3** ❌ | ❌ | error |
| `(3, 4)` | `(3, 1)` | 4 vs 1 ✅, 3 vs 3 ✅ | ✅ | `(3, 4)` |

The third row is a trap. Suppose each **store** pays a fixed daily rent, one number per store:

In [ ]:
rent_per_store = np.array([50, 40, 70])     # Store A, B, C -> shape (3,)

try:
    net - rent_per_store
except ValueError as err:
    print("ValueError:", err)

The shapes `(3, 4)` and `(3,)` don't match from the right (4 vs 3). NumPy refuses **even though we "know" the 3 means stores**. Remember, NumPy only knows positions.

To use a per-store number on every product, make the rent a **column**, shape `(3, 1)`. Then 4 vs 1 works, and 3 vs 3 works:

In [ ]:
rent_column = rent_per_store.reshape(3, 1)
show(rent_column)

(Rent is a daily cost for the whole store, so here we just spread it evenly across the four products.)

In [ ]:
net_after_rent = net - rent_column / 4
show(net_after_rent)

**Takeaway:** before combining arrays of different shapes, write down both shapes and check them from the right. If they don't fit, NumPy raises an error, which is good. The dangerous case is when shapes happen to fit but mean different things, for example a table with 4 stores and 4 products. Then NumPy computes something, just not what you meant.

## 1.5 Reductions: operations that collapse an axis

A **reduction** turns many values into fewer: `sum`, `mean`, `min`, `max`, `std`, `argmax`, … Each can work on the whole array or along one axis.

Twelve weeks of sales for our three stores, now arranged **weeks × stores** (one row per week):

In [ ]:
weekly_sales = np.array([
    # A   B   C
    [10,  8, 20],
    [12,  9, 18],
    [15, 14, 17],
    [11, 13, 22],
    [14, 10, 19],
    [13, 12, 21],
    [16, 11, 23],
    [12, 15, 20],
    [17, 13, 24],
    [15, 14, 22],
    [18, 12, 25],
    [16, 16, 23],
])
print(weekly_sales.shape)

🔮 **Predict** the shape of each result. Which one gives "the average week for each store"?

In [ ]:
show(weekly_sales.mean(axis=0))   # collapse the 12 weeks -> one average per store

In [ ]:
show(weekly_sales.mean(axis=1))   # collapse the 3 stores -> one average per week

In [ ]:
show(weekly_sales.max())          # the single best store-week

Here the stores are **columns**, so the per-store average uses `axis=0`. In §1.1 the stores were **rows**, and the per-store total used `axis=1`. The axis number only means something if you know how the array is laid out, so check the shape first.

More reductions:

In [ ]:
print("best week per store (position):", weekly_sales.argmax(axis=0))
print("spread of each store (std):    ", weekly_sales.std(axis=0).round(2))
print("weeks above 20 units, per store:", (weekly_sales > 20).sum(axis=0))

The last line combines two ideas. `weekly_sales > 20` is an elementwise comparison, a `(12, 3)` array of True/False. Summing it counts the `True` values per column.

**✏️ Try it:** What was the **total** number of units sold by all stores in each week? Predict the shape first.

## 1.6 Reshaping

Store A recorded its sales **every day** for four weeks, 28 numbers in a row:

In [ ]:
daily_sales = np.array([
    52, 48, 50, 55, 70, 88, 81,    # week 1: Mon ... Sun
    50, 47, 53, 57, 72, 90, 85,    # week 2
    49, 51, 52, 58, 75, 93, 80,    # week 3
    55, 50, 54, 60, 78, 95, 87,    # week 4
])
show(daily_sales)

To compare weekdays, we want a table with **one row per week and one column per weekday**. `reshape` reorganizes the same 28 values into 4 rows of 7:

In [ ]:
by_week = daily_sales.reshape(4, 7)
show(by_week)

`reshape` fills the new shape **row by row**: the first 7 values become row 0, the next 7 row 1, and so on. That works here only because the data was recorded in exactly that order: Monday to Sunday, week after week.

Now the axis operations from §1.5 answer real questions:

In [ ]:
weekdays = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

print("average per weekday:", by_week.mean(axis=0).round(1))
print("total per week:     ", by_week.sum(axis=1))
print("busiest weekday:    ", weekdays[by_week.mean(axis=0).argmax()])

**What reshape does not do:**

- It doesn't **add up** or **average** anything. The 28 values are all still there, just arranged differently.
- It can't **invent** values. The new shape must hold exactly the same number of values:

In [ ]:
print(by_week.size, "values")

try:
    daily_sales.reshape(5, 6)       # 30 places for 28 values
except ValueError as err:
    print("ValueError:", err)

You can let NumPy work out one length by writing `-1`: `daily_sales.reshape(-1, 7)` means "rows of 7, as many rows as needed". `.ravel()` flattens an array back into one long vector:

In [ ]:
show(daily_sales.reshape(-1, 7).shape)
show(by_week.ravel()[:10])      # the first 10 values of the flattened array

## 1.7 Conditional arrays: masks and `np.where`

A **mask** is an array of True/False, created by an elementwise comparison. You met masks in the first notebook. Here we use them to apply a **business rule** to every element.

`margin` below is the profit per unit for each store (rows) and product (columns). Negative means we lose money on that product in that store:

In [ ]:
margin = np.array([
    # espresso latte muffin bagel
    [ 2.20,    3.30,  0.40, -0.15],   # Store A
    [ 2.00,    3.10, -0.20,  0.10],   # Store B
    [ 2.40,    3.50,  0.60, -0.30],   # Store C
])

losing = margin < 0
show(losing)

In [ ]:
print("losing combinations:", losing.sum())
print("any loss in Store A?", losing[0].any())
print("all espresso profitable?", (margin[:, 0] > 0).all())

`margin[losing]` pulls out only the negative values. The result is a **1-D** array, because the selected values don't form a rectangle anymore:

In [ ]:
show(margin[losing])

### `np.where(condition, if_true, if_false)`

`np.where` applies a rule to **every element** and keeps the shape. Think of it as an `if`/`else` for a whole array:

In [ ]:
labels = np.where(margin < 0, "loss", "profit")
show(labels)

The choices can also be arrays or computations. For example, a reorder rule: if stock is below the reorder level, order enough to get back to that level, otherwise order nothing.

In [ ]:
stock         = np.array([30, 5, 12, 0])
reorder_level = np.array([20, 20, 10, 15])

to_order = np.where(stock < reorder_level, reorder_level - stock, 0)
for p, n in zip(products, to_order):
    print(f"{p:9} order {n}")

## 1.8 A brief caution: views versus copies

When you take a **basic slice** (with `:`) of an array, NumPy usually doesn't copy the data. The slice is a **view**, a window onto the **same** numbers. Changing the view changes the original:

In [ ]:
a = np.array([10, 20, 30, 40, 50])
part = a[:3]           # basic slice -> a view

part[0] = 999
print("part:", part)
print("a:   ", a)      # the original changed too!

Selecting with a **list of positions** or a **mask** (called *advanced indexing*) always makes a **copy**. Changing it leaves the original alone:

In [ ]:
a = np.array([10, 20, 30, 40, 50])
part = a[[0, 2]]       # list of positions -> a copy

part[0] = 999
print("part:", part)
print("a:   ", a)      # unchanged

The practical question is always: **will changing this affect the original?** If you plan to modify a slice but want to keep the original intact, make the copy explicit with `.copy()`:

In [ ]:
a = np.array([10, 20, 30, 40, 50])
part = a[:3].copy()    # now a real, independent copy

part[0] = 999
print("part:", part)
print("a:   ", a)      # unchanged

### 🧩 Exercises — NumPy

Use this array of **units sold**: 4 stores (rows) × 3 products (columns: coffee, tea, pastry).

```python
units = np.array([
    [120, 40, 60],
    [ 90, 55, 45],
    [150, 30, 80],
    [ 60, 70, 20],
])
price = np.array([3.50, 3.00, 2.75])
```

For each step, **write your predicted shape as a comment first**, then run it.

1. What is `units.shape`? Get store 2's sales of all products, and the tea sales of all stores.
2. Compute `revenue`, the revenue per store and product. Which NumPy feature makes `units * price` work?
3. Total revenue **per store** and **per product**. Which axis is which?
4. Each store has a daily fixed cost: `fixed = np.array([200, 180, 260, 150])`. Compute each store's profit, total revenue minus fixed cost. (Hint: reduce first, then subtract. Do the shapes match?)
5. Use `np.where` to label each store `"above target"` if its total revenue is at least 700, otherwise `"below target"`.
6. Make `top = units[:2]` and set `top[0, 0] = 0`. Did `units` change? How would you prevent that?

In [ ]:
units = np.array([
    [120, 40, 60],
    [ 90, 55, 45],
    [150, 30, 80],
    [ 60, 70, 20],
])
price = np.array([3.50, 3.00, 2.75])

# 1.

# 2.

# 3.

# 4.

# 5.

# 6.

---

# Part 2 — pandas: labels and reshaping

A NumPy array knows only positions. pandas adds three things:

- **Labels** for rows (the *index*) and columns (column names).
- Columns of **different types** in one table: text, numbers, dates.
- Tools for **dates and time**.

Under the hood, each pandas column is still a NumPy array.

This week's summary for our three stores (plus a new store D that opened recently):

In [ ]:
stores = pd.DataFrame({
    "store":   ["A", "B", "C", "D"],
    "region":  ["North", "South", "North", None],
    "sales":   [4200.0, 3800.0, 6100.0, 1500.0],
    "costs":   [3100.0, 3300.0, 4000.0, 1700.0],
    "opened":  pd.to_datetime(["2019-03-01", "2021-06-15", "2017-09-10", "2026-08-01"]),
})
stores

The grain: **one row per store**. The columns have different types (text, decimal numbers, dates), which a single NumPy array couldn't hold efficiently. Store D's region hasn't been entered yet, so it's missing.

## 2.1 `Series` versus `DataFrame`

🔮 **Predict:** what is the difference between these two lines?

In [ ]:
one = stores["sales"]
one

In [ ]:
two = stores[["sales"]]
two

In [ ]:
print(type(one), one.shape)
print(type(two), two.shape)

- `stores["sales"]` is **one name** in brackets. It returns a **Series**, a 1-dimensional labeled column, shape `(4,)`. It's like a NumPy vector with labels.
- `stores[["sales"]]` is a **list of names** in brackets. It returns a **DataFrame** with one column, shape `(4, 1)`. It's like a matrix with one column.

This mirrors `sales[:, 0]` versus `sales[:, 0:1]` in NumPy (§1.2).

The difference matters because the two types behave differently downstream:

In [ ]:
print(stores["sales"].mean())      # Series -> one number
print()
print(stores[["sales"]].mean())    # DataFrame -> a Series with one value per column

Some tools want a Series (e.g., a single column for a chart axis). Others want a DataFrame (e.g., `merge`, or writing to CSV with headers). If code complains about the type, check how many brackets you used. You can convert with `series.to_frame()`, and for a one-column DataFrame, `df["col"]` gets the Series back.

## 2.2 Position versus label: `.iloc` and `.loc`

Every DataFrame has two ways to point at rows and columns:

| | `.iloc[...]` | `.loc[...]` |
|---|---|---|
| uses | integer **positions** (0, 1, 2, …) | row **labels** and column **names** |
| like | NumPy indexing | looking something up by name |
| slice `a:b` | **excludes** `b` (like Python) | **includes** `b` |

Let's use the store name as the row label. `set_index` moves a column into the index:

In [ ]:
by_store = stores.set_index("store")
by_store

Now the rows have labels A, B, C, D. The next two lines ask for the same row in two ways:

In [ ]:
print(by_store.iloc[2])     # the row at POSITION 2 (third row)
print()
print(by_store.loc["C"])    # the row LABELED "C"

Rows **and** columns, by position and by name:

In [ ]:
by_store.iloc[:3, :2]              # first 3 rows, first 2 columns (positions 0, 1)

In [ ]:
by_store.loc[:, ["region", "sales"]]   # all rows, columns by name

### The slice surprise

🔮 **Predict:** how many rows does each line return?

In [ ]:
print(by_store.iloc[0:2])      # positions 0 and 1 -- stop is EXCLUDED
print()
print(by_store.loc["A":"C"])   # labels A, B, and C -- stop is INCLUDED

`.iloc` follows Python's exclusive-stop rule, but `.loc` **includes** the end label. It's a deliberate choice: with labels, "from A to C" naturally means C too, and there's often no "next label" to stop at.

### A label 5 is not the sixth row

With the default index 0, 1, 2, … labels and positions look identical, and that hides the difference. It shows up as soon as you sort or filter. Sort the original `stores` by sales:

In [ ]:
ranked = stores.sort_values("sales", ascending=False)
ranked

The index labels travelled with their rows: the labels now read 2, 0, 1, 3. 🔮 **Predict:** which store does each line return?

In [ ]:
print(ranked.iloc[0]["store"])   # position 0 -> the first row shown -> the top seller
print(ranked.loc[0]["store"])    # label 0     -> the row labeled 0 -> Store A

`.iloc[0]` means "the first row as it is displayed now". `.loc[0]` means "the row whose label is 0, wherever it is". If you want the top seller, you need `.iloc[0]`.

`.loc` also takes a mask for the rows and a list of names for the columns. It's the standard way to filter and select in one step:

In [ ]:
stores.loc[stores["sales"] > 4000, ["store", "sales"]]

**✏️ Try it:** From `by_store`, get the `costs` of stores B **through** D with `.loc`. Then get the same values with `.iloc`. (Which positions? Which column position?)

## 2.3 Automatic alignment

NumPy combines two arrays **by position**: element 0 with element 0. pandas combines two Series **by label**: "A" with "A", wherever they are.

Sales per store from the point-of-sale system, and costs per store from accounting. Accounting lists the stores in a different order, and it has no record for store B yet but does have one for a new store E:

In [ ]:
sales_by_store = pd.Series([4200, 3800, 6100], index=["A", "B", "C"])
costs_by_store = pd.Series([4000, 3100, 900], index=["C", "A", "E"])

print(sales_by_store, "\n")
print(costs_by_store)

🔮 **Predict** before you run it: how many rows will the result have? What will B and E show?

In [ ]:
profit = sales_by_store - costs_by_store
profit

pandas **matched the labels**, not the positions:

- A: 4200 − 3100 = 1100, even though A is first in one Series and second in the other.
- C: 6100 − 4000 = 2100.
- B has no cost, and E has no sales. The result is `NaN`: **unknown**, not zero.

The result contains **every label from both sides**, sorted. That is exactly what a **full outer join on the index** would give you in SQL.

Compare with NumPy, which pairs by position and silently computes nonsense:

In [ ]:
print(sales_by_store.to_numpy() - costs_by_store.to_numpy())   # A-C, B-A, C-E ... wrong pairs!

When labels **identify the same things** (store names, product IDs, dates), alignment is a safety feature. You can't accidentally subtract store C's costs from store A's sales.

If an unknown value really should count as zero, say so explicitly with `fill_value`:

In [ ]:
sales_by_store.sub(costs_by_store, fill_value=0)

### When alignment bites: accidental indexes

Alignment is **hazardous when the index is accidental**, for example the leftover 0, 1, 2, … labels after filtering.

Get the sales of the North stores, then add a bonus for each of them. The bonuses come from a list:

In [ ]:
north_sales = stores.loc[stores["region"] == "North", "sales"]
bonus = pd.Series([100, 200])          # one bonus per North store, in order

print(north_sales, "\n")
print(bonus)

In [ ]:
north_sales + bonus

`north_sales` kept its **original labels 0 and 2**. The new `bonus` Series got fresh labels **0 and 1**. pandas matched label 0 with label 0, left label 2 without a bonus, and label 1 without a sale. We wanted positions, but pandas used labels.

Two fixes, depending on what you mean:

In [ ]:
# 1. "Pair them in order": drop the old labels on the left
north_sales.reset_index(drop=True) + bonus

In [ ]:
# 2. Better: give the bonus a real label, so the pairing can't go wrong
north_by_store = by_store.loc[by_store["region"] == "North", "sales"]
bonus_by_store = pd.Series({"C": 200, "A": 100})      # order doesn't matter anymore

north_by_store + bonus_by_store

The second version is safer. Each bonus now carries the name of the store it belongs to. Labels that **mean something** make alignment work for you.

## 2.4 Wide ↔ long reshaping

A manager sends monthly sales as a spreadsheet: one row per store, one column per month. This is the **wide** layout:

In [ ]:
wide = pd.DataFrame({
    "store": ["A", "B", "C"],
    "Jan":   [100,  80, 150],
    "Feb":   [120,  95, 160],
    "Mar":   [130,  90, 170],
})
wide

The wide layout is easy for people to read, but awkward for analysis. "Average sales per month" means averaging **columns**. "Which month was best?" compares column **names**. Adding April means changing the table's structure.

### `melt`: wide → long

`melt` turns the month columns into rows. Each (store, month) pair becomes one row:

In [ ]:
long = wide.melt(
    id_vars="store",       # column(s) to keep as identifiers
    var_name="month",      # new column that holds the old column NAMES
    value_name="sales",    # new column that holds the values
)
long

State the grain again:

- `wide`: **one row per store**, shape (3, 4).
- `long`: **one row per store–month**, shape (9, 3).

The 9 numbers are the same. Only the layout changed, like `reshape` in NumPy, but driven by labels.

In long form, the questions become ordinary grouping, which you already know:

In [ ]:
long.groupby("month", as_index=False, sort=False)["sales"].sum()

In [ ]:
long.loc[long["sales"] == long["sales"].max()]

Long data also fits SQL tables, and plotting tools like it: one column for x (`month`), one for the value (`sales`), one for color (`store`).

### `pivot`: long → wide

`pivot` goes the other way. Pick which column becomes the **rows** (`index`), which one supplies the new **column names** (`columns`), and which one fills the **cells** (`values`):

In [ ]:
back_to_wide = long.pivot(index="store", columns="month", values="sales")
back_to_wide

pandas sorted the new columns alphabetically (Feb, Jan, Mar). Choose the order by selecting the columns: `back_to_wide[["Jan", "Feb", "Mar"]]`.

### When `pivot` fails: duplicates

`pivot` needs **exactly one value per output cell**. Suppose the long data has two entries for store A in January, say two separate transactions:

In [ ]:
transactions = pd.DataFrame({
    "store": ["A", "A", "B", "A"],
    "month": ["Jan", "Jan", "Jan", "Feb"],
    "sales": [60, 40, 80, 120],
})

try:
    transactions.pivot(index="store", columns="month", values="sales")
except ValueError as err:
    print("ValueError:", err)

pandas can't put two numbers into one cell, and it won't guess what you want. `pivot_table` can **aggregate** the duplicates. You say how with `aggfunc`:

In [ ]:
transactions.pivot_table(index="store", columns="month", values="sales", aggfunc="sum")

Store A's January is 60 + 40 = 100. Store B has no February entry, so its cell is `NaN`. `pivot_table` is a **group-by and a reshape in one step**. The grain went from *one row per transaction* to *one row per store*.

**✏️ Try it:** Change `aggfunc="sum"` to `"count"`. What does the table show now?

## 2.5 Dates, lags, and resampling

Many business questions are about **change over time**: *compared with yesterday? monthly totals? the trend without daily noise?* pandas is especially good at these.

Daily sales for store A over 90 days. The numbers are generated with a simple formula (a weekly pattern, a slow upward trend, and some random noise with a fixed seed), so everyone gets the same values:

In [ ]:
rng = np.random.default_rng(seed=8410)

days = pd.date_range("2026-01-01", periods=90, freq="D")
weekend_boost = np.where(days.dayofweek >= 5, 30, 0)          # Sat/Sun sell more
trend = np.arange(90) * 0.3                                    # slowly growing
noise = rng.integers(-8, 9, size=90)

daily = pd.Series(400 + weekend_boost + trend + noise, index=days, name="sales").round()
daily.head(10)

The index is now made of **dates** (a `DatetimeIndex`). Dates as labels let you select by date. A partial date string selects a whole period:

In [ ]:
daily.loc["2026-02-10"]              # one day

In [ ]:
daily.loc["2026-02-01":"2026-02-07"]   # a week; like all .loc slices, the end is included

In [ ]:
print(daily.loc["2026-03"].sum())    # all of March

### Lags with `shift`: compare with the previous period

`shift(1)` moves every value **one step later**. Each day then sees the previous day's value in the same row:

In [ ]:
compare = pd.DataFrame({
    "sales":      daily,
    "yesterday":  daily.shift(1),
})
compare["change"]   = compare["sales"] - compare["yesterday"]     # the same as daily.diff()
compare["pct_change"] = (compare["change"] / compare["yesterday"] * 100).round(1)
compare.head(8)

The first day has no "yesterday", so it gets `NaN`, not zero. `shift(7)` would compare with the same weekday last week, which is often fairer for sales with a weekly pattern.

**Caution with several stores:** in a long table with several stores, a plain `shift(1)` would give the first day of store B the last day of store A. Shift **within each store** instead: `df.groupby("store")["sales"].shift(1)`. It's the pandas version of `LAG(...) OVER (PARTITION BY store ...)` from the previous notebook.

### Resampling: change the time grain

`resample` groups by **time periods**. It's a `groupby` for dates. The code says which period: `"W"` = week, `"MS"` = month (labeled by its first day, *month start*):

In [ ]:
weekly = daily.resample("W").sum()
weekly.head()

Look at the first row. Weeks in pandas end on **Sunday** by default, and 2026 began on a Thursday. So the first "week" holds only four days (Jan 1–4), and its total looks low. **Partial periods at the edges** of a date range are a classic trap. Check them before comparing periods.

In [ ]:
monthly = daily.resample("MS").sum()
monthly

The grain changed from **one row per day** (90 rows) to **one row per week** or **one row per month** (3 rows). Resampling combines a date index and a reduction in one step.

February looks weaker than January, but it has only 28 days against January's 31. Totals over periods of **different length** aren't directly comparable. Dividing by the number of days (`daily.resample("MS").mean()`) gives a fairer comparison.

`resample` plus `diff` answers "how did each month compare with the previous one?":

In [ ]:
pd.DataFrame({"sales": monthly, "change_vs_prev_month": monthly.diff()})

### Rolling averages: smooth out the noise

A **rolling** calculation uses a moving window of recent rows. A 7-day rolling mean averages each day with the 6 before it. That removes the weekday pattern and shows the trend:

In [ ]:
smooth = pd.DataFrame({
    "sales":      daily,
    "rolling_7d": daily.rolling(7).mean().round(1),
})
smooth.iloc[4:12]

Unlike `resample`, rolling keeps **one row per day**. It's a window calculation, the grain doesn't change. The first 6 days are `NaN` because a full 7-day window isn't available yet.

**✏️ Try it:** Plot the smoothed series with `smooth.plot()`. Can you see the weekend spikes in `sales` and the trend in `rolling_7d`?

## 2.6 Categorical data and missing values

Back to the store table. One store has no region:

In [ ]:
stores[["store", "region"]]

**Missing is not a category.** `None`/`NaN` means *we don't know the region*. That's different from a real value like `"Other"`, which means *we know, and it's none of the listed regions*. Treat them differently.

**Check missing values before you compute statistics or draw charts.** Many pandas functions skip missing values without telling you:

In [ ]:
print(stores["region"].isna())
print("missing regions:", stores["region"].isna().sum())

In [ ]:
print(stores["region"].value_counts())                  # hides the missing store!
print()
print(stores["region"].value_counts(dropna=False))      # shows it

The first count adds up to 3, but we have 4 stores. Store D disappeared silently. Group-bys do the same by default:

In [ ]:
stores.groupby("region")["sales"].sum()                  # store D's sales are missing

In [ ]:
stores.groupby("region", dropna=False)["sales"].sum()    # keep the unknown group visible

If you decide to replace the missing value, use a label that says **unknown**, and make it a deliberate, visible step. Don't quietly put it into an existing group like `"Other"`:

In [ ]:
stores["region_filled"] = stores["region"].fillna("Unknown")
stores[["store", "region", "region_filled"]]

### Categorical columns

A column with a small, fixed set of values (regions, sizes, ratings) can be stored as a **categorical** type. It lists the allowed values, and it can give them a meaningful **order**:

In [ ]:
cups = pd.DataFrame({
    "order": [1, 2, 3, 4, 5, 6],
    "size":  ["Large", "Small", "Medium", "Small", "Large", "Medium"],
})

cups["size"] = pd.Categorical(cups["size"], categories=["Small", "Medium", "Large"], ordered=True)
cups.sort_values("size")          # sorted Small < Medium < Large, not alphabetically

In [ ]:
print(cups["size"].cat.categories)
print((cups["size"] >= "Medium").sum(), "orders were Medium or larger")

Without the categorical type, sorting would be alphabetical: Large, Medium, Small.

### 🧩 Exercises — pandas

Use this table of **quarterly revenue** (in $1000) per product line:

```python
quarterly = pd.DataFrame({
    "product": ["coffee", "tea", "pastry", "merch"],
    "category": ["drinks", "drinks", "food", None],
    "Q1": [120, 45, 60, 10],
    "Q2": [130, 50, 58, 12],
    "Q3": [125, 40, 70, 15],
    "Q4": [150, 60, 75, 30],
})
```

1. What does `quarterly["Q1"]` return, and what does `quarterly[["Q1"]]` return? Give the type and the shape of each.
2. Set `product` as the index. Use `.loc` to get Q2 **through** Q4 for tea and pastry. Then get the same values with `.iloc`.
3. `melt` the table into long form with columns `product`, `category`, `quarter`, and `revenue`. What is the grain of the result, and how many rows do you expect?
4. With the long table, compute total revenue per quarter, then the change from the previous quarter (`diff`).
5. How much revenue per `category`? Make sure `merch` (missing category) is not silently dropped.
6. **Bonus:** A second Series holds each product's target for the year: `pd.Series({"tea": 200, "coffee": 500, "merch": 50})`. Subtract it from each product's yearly total. What happens to `pastry`, and why?

In [ ]:
quarterly = pd.DataFrame({
    "product": ["coffee", "tea", "pastry", "merch"],
    "category": ["drinks", "drinks", "food", None],
    "Q1": [120, 45, 60, 10],
    "Q2": [130, 50, 58, 12],
    "Q3": [125, 40, 70, 15],
    "Q4": [150, 60, 75, 30],
})

# 1.

# 2.

# 3.

# 4.

# 5.

# 6.

---

# Summary: position, label, or join key?

Every operation in this notebook, and in the previous one, combines values in one of three ways:

| Combined by | Tool | Example | What can go wrong |
|---|---|---|---|
| **position** | NumPy | `units * prices`: element 0 with element 0 | shapes happen to fit but mean different things |
| **label** | pandas | `sales_by_store - costs_by_store`: "A" with "A" | an accidental index (0, 1, 2, … after filtering) pairs the wrong rows |
| **join key** | SQL, `merge` | `orders JOIN customers ON customerid` | a wrong or incomplete key, duplicate keys |

And the four questions to ask before running any operation:

1. What does one input element or row represent?
2. What is the input shape or table grain?
3. What will the output shape or grain be?
4. Are values combined by position, by label, or by a join key?

### Key points

**NumPy**

- `shape` describes an array. `axis=0` works down the rows (one result per column), `axis=1` across the columns (one result per row). The named axis disappears.
- A single index removes an axis, and a slice keeps it: `a[:, 0]` is `(n,)`, `a[:, 0:1]` is `(n, 1)`.
- `*` is elementwise, `@` is multiply-and-add.
- Broadcasting compares shapes from the right. Lengths must match or be 1.
- `reshape` reorganizes values without changing them. `np.where` applies a rule to every element.
- Basic slices are views, so changes reach the original. Use `.copy()` when in doubt.

**pandas**

- `df["col"]` → Series (1-D). `df[["col"]]` → DataFrame (2-D).
- `.iloc` = positions (stop excluded). `.loc` = labels (stop **included**).
- Arithmetic on Series aligns by **label**, like an outer join on the index. Missing partners give `NaN`.
- `melt` (wide → long) and `pivot` (long → wide) change the layout and the grain. `pivot_table` aggregates duplicates.
- A date index enables `.loc["2026-03"]`, `shift` (lags), `resample` (new time grain), and `rolling` (moving windows).
- Missing is not a category. Check `isna()` and use `dropna=False` before you trust counts, groups, or charts.

### Where to go next

We skipped some powerful tools on purpose: advanced broadcasting tricks, NumPy strides and memory layout, pandas `MultiIndex`, `stack`/`unstack`, long chained indexing, and most of the time-series API. You will meet some of them when a real problem needs them. For reference:

- NumPy: [Indexing on ndarrays](https://numpy.org/doc/stable/user/basics.indexing.html) and [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- pandas: [Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html), [Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html), and [Time series](https://pandas.pydata.org/docs/user_guide/timeseries.html)